# WaveFollower — Validation & Backtesting
### Load from Google Drive → Evaluate → Realistic Backtest ($100 start, 10% risk, OANDA conditions)

**WaveFollower** is pair-agnostic trend-following model trained on 4 pairs (GBP/JPY, EUR/JPY, USD/JPY, GBP/USD).

This notebook:
1. Mounts Google Drive and loads the trained WaveFollower checkpoint
2. Runs holdout test evaluation (accuracy, per-class F1, confusion matrix)
3. Backtests with **realistic OANDA conditions**: $100 starting balance, 10% risk/trade, real spreads, 30:1 leverage
4. Full trade breakdown: daily/weekly/monthly/yearly, session analysis
5. Realistic friction simulation: slippage, spread widening, lot caps

---
| OANDA Reality | Value |
|---------------|-------|
| Starting Balance | $100 |
| Risk per Trade | 10% ($10 at start) |
| Leverage | 30:1 (ESMA retail) |
| GBP/JPY spread | 2.5 pips (OANDA avg) |
| EUR/JPY spread | 2.0 pips |
| USD/JPY spread | 1.4 pips |
| GBP/USD spread | 1.6 pips |
| Commission | $0 (OANDA Standard) |
| Min lot | 0.01 (1,000 units) |
| OANDA Account | `101-001-30902818-002` |

## 1. Setup & Mount Google Drive

In [ ]:
# ── Workspace Setup ───────────────────────────────────────────────────────────
from google.colab import drive
import os, pathlib, sys

drive.mount("/content/drive")

DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/phase_lm")
DRIVE_CKPT = DRIVE_ROOT / "checkpoints"
DRIVE_PROC = DRIVE_ROOT / "processed_data"

LOCAL_ROOT = pathlib.Path("/content/phase_lm")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

print("Google Drive mounted.")
print(f"Checkpoints dir  : {DRIVE_CKPT}")
print(f"Processed data   : {DRIVE_PROC}")

# Show available WaveFollower checkpoints
wf_ckpts = sorted(DRIVE_CKPT.glob("wavefollower_*"), key=lambda p: p.name, reverse=True)
print(f"\nWaveFollower checkpoints found: {len(wf_ckpts)}")
for c in wf_ckpts[:5]:
    print(f"  {c.name}")

## 2. Clone Repository & Install Dependencies

In [ ]:
import subprocess, shutil

REPO_URL = "https://github.com/Unseengap/wavetrader.git"
REPO_DIR = "/content/phase_lm"

if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    print("Cloning repository...")
    if os.path.exists(REPO_DIR):
        os.chdir("/content")
        shutil.rmtree(REPO_DIR, ignore_errors=True)
    os.chdir("/content")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("Repository exists. Pulling latest...")
    os.chdir(REPO_DIR)
    subprocess.run(["git", "stash"], check=False)
    subprocess.run(["git", "pull", "origin", "main"], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# Install deps
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "ta", "pandas_ta", "scikit-learn", "scipy",
                "fastparquet", "pyarrow"], check=False)
if os.path.exists("requirements.txt"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Setup complete.")

## 3. Load WaveFollower Model from Google Drive

In [ ]:
import torch, json, hashlib
from wavetrader.wave_follower import WaveFollower, WaveFollowerConfig

# ── Select checkpoint (latest wavefollower_* dir) ─────────────────────────────
wf_dirs = sorted(DRIVE_CKPT.glob("wavefollower_*"), key=lambda p: p.name, reverse=True)
if not wf_dirs:
    raise RuntimeError(f"No WaveFollower checkpoints in {DRIVE_CKPT}")

CKPT_DIR = wf_dirs[0]
print(f"Loading checkpoint: {CKPT_DIR.name}")

# ── Verify SHA-256 integrity ───────────────────────────────────────────────────
weights_path = CKPT_DIR / "model_weights.pt"
sha_path = CKPT_DIR / "weights.sha256"

if sha_path.exists():
    expected = sha_path.read_text().split()[0]
    h = hashlib.sha256()
    with open(weights_path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    actual = h.hexdigest()
    if expected == actual:
        print(f"SHA-256 verified: {actual[:24]}...")
    else:
        raise RuntimeError(f"Checksum MISMATCH! expected={expected[:24]}, actual={actual[:24]}")

# ── Load config ────────────────────────────────────────────────────────────────
config_path = CKPT_DIR / "config.json"
if config_path.exists():
    with open(config_path) as f:
        cfg_dict = json.load(f)
    clean = {k: v for k, v in cfg_dict.items()
             if not k.startswith("_") and k in WaveFollowerConfig.__dataclass_fields__}
    config = WaveFollowerConfig(**clean)
    print(f"Config loaded: {config.timeframes}, fused_dim={config.fused_dim}")
else:
    config = WaveFollowerConfig()
    print("No config.json found — using defaults")

# ── Instantiate & load weights ─────────────────────────────────────────────────
model = WaveFollower(config).to(device)

# Handle full checkpoint (dict with model_state_dict) or bare state_dict
checkpoint = torch.load(str(weights_path), map_location=device, weights_only=False)
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint

# Strip _orig_mod. prefix from torch.compile'd checkpoints
cleaned = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
model.load_state_dict(cleaned)

if isinstance(checkpoint, dict):
    train_info = checkpoint.get("training", {})
    print(f"Best val acc: {train_info.get('best_val_accuracy', 'N/A')}")
    print(f"Test acc    : {train_info.get('test_accuracy', 'N/A')}")

model.eval()
print(f"\nWaveFollower loaded: {model.count_parameters():,} params on {device}")

## 4. Load Test Data (All 4 Pairs)

In [ ]:
import pandas as pd
import torch.utils.data as tud
from wavetrader.dataset import MTFForexDataset, mtf_collate_fn
from torch.utils.data import ConcatDataset

PAIRS = ["GBPJPY", "EURJPY", "USDJPY", "GBPUSD"]
PAIR_NAMES = {"GBPJPY": "GBP/JPY", "EURJPY": "EUR/JPY", "USDJPY": "USD/JPY", "GBPUSD": "GBP/USD"}
TFS = ["15m", "1h", "4h", "1d"]
TF_KEY = {"15m": "15min", "1h": "1h", "4h": "4h", "1d": "1d"}

# OANDA typical spreads per pair (pips)
OANDA_SPREADS = {"GBPJPY": 2.5, "EURJPY": 2.0, "USDJPY": 1.4, "GBPUSD": 1.6}
# Pip value per standard lot (USD)
PIP_VALUES = {"GBPJPY": 6.5, "EURJPY": 6.5, "USDJPY": 6.5, "GBPUSD": 10.0}

def load_pair_test(pair_tag):
    """Load all 4 TF parquets for a pair's test split."""
    dfs = {}
    for tf in TFS:
        path = DRIVE_PROC / "test" / f"{pair_tag}_{tf}.parquet"
        if not path.exists():
            print(f"  WARNING: {path.name} not found")
            return None
        df = pd.read_parquet(path)
        df.index.name = "date"
        df = df.reset_index()
        df = df.rename(columns={
            "open_mid": "open", "high_mid": "high",
            "low_mid": "low", "close_mid": "close",
        })
        dfs[TF_KEY[tf]] = df
    return dfs

# Load all pairs
test_data = {}  # pair_tag -> {tf: df}
test_datasets = []  # for combined evaluation
per_pair_datasets = {}  # for per-pair backtesting

print("Loading test data from Drive...")
for pair_tag in PAIRS:
    dfs = load_pair_test(pair_tag)
    if dfs is None:
        print(f"  {pair_tag}: SKIPPED (missing data)")
        continue
    test_data[pair_tag] = dfs
    ds = MTFForexDataset(dfs, config, pair=PAIR_NAMES[pair_tag])
    test_datasets.append(ds)
    per_pair_datasets[pair_tag] = ds
    entry_bars = len(dfs[config.entry_timeframe])
    print(f"  {pair_tag}: {entry_bars:,} entry bars, {len(ds):,} samples")

# Combined dataset (all pairs) for overall evaluation
combined_test = ConcatDataset(test_datasets)
test_loader = tud.DataLoader(
    combined_test, batch_size=128, shuffle=False,
    collate_fn=mtf_collate_fn, num_workers=2, pin_memory=True,
)

print(f"\nCombined test: {len(combined_test):,} samples, {len(test_loader)} batches")

## 5. Holdout Validation — Accuracy, F1, Confusion Matrix

In [ ]:
import time
import numpy as np
from wavetrader.train_wave_follower import TrendFollowerLoss

criterion = TrendFollowerLoss().to(device)
model.eval()

total_loss, correct, n = 0.0, 0, 0
num_batches = len(test_loader)

print(f"Evaluating WaveFollower on combined test set ({num_batches} batches)...")
t0 = time.time()

with torch.no_grad():
    for i, batch in enumerate(test_loader):
        labels = batch["label"].to(device)
        model_input = {
            tf: {k: v.to(device) for k, v in batch[tf].items()}
            for tf in config.timeframes
            if tf in batch and isinstance(batch[tf], dict)
        }
        out = model(model_input)
        losses = criterion(out, labels)
        total_loss += losses["total"].item()
        correct += (out["signal_logits"].argmax(-1) == labels).sum().item()
        n += labels.size(0)

        if (i + 1) % max(1, num_batches // 10) == 0 or (i + 1) == num_batches:
            print(f"  [{(i+1)/num_batches*100:5.1f}%]  {n:,} samples  |  acc: {correct/max(n,1)*100:.2f}%  |  {time.time()-t0:.1f}s")

val_loss = total_loss / max(num_batches, 1)
val_acc = correct / max(n, 1)

print(f"\n{'='*50}")
print(f"  WaveFollower Holdout Results")
print(f"{'='*50}")
print(f"  Loss     : {val_loss:.4f}")
print(f"  Accuracy : {val_acc:.4f} ({val_acc*100:.2f}%)")
print(f"  Samples  : {n:,} (all 4 pairs combined)")
print(f"  Time     : {time.time()-t0:.1f}s")
print(f"{'='*50}")

In [ ]:
# ── Detailed metrics: F1, confusion matrix, confidence calibration ─────────
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import precision_recall_fscore_support
import matplotlib.pyplot as plt

all_preds, all_labels, all_confs, all_trends = [], [], [], []

print(f"Collecting predictions for visual analysis...")
t0 = time.time()

with torch.no_grad():
    for batch in test_loader:
        labels = batch["label"].to(device)
        model_input = {
            tf: {k: v.to(device) for k, v in batch[tf].items()}
            for tf in config.timeframes
            if tf in batch and isinstance(batch[tf], dict)
        }
        out = model(model_input)
        all_preds.append(out["signal_logits"].argmax(-1).cpu())
        all_labels.append(labels.cpu())
        all_confs.append(out["confidence"].squeeze(-1).cpu())
        all_trends.append(out["trend_logits"].argmax(-1).cpu())

all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()
all_confs = torch.cat(all_confs).numpy()
all_trends = torch.cat(all_trends).numpy()

CLASS_NAMES = ["BUY", "SELL", "HOLD"]
TREND_NAMES = ["UP", "DOWN", "NEUTRAL"]
accuracy = (all_preds == all_labels).mean() * 100

print(f"\nOverall Accuracy: {accuracy:.2f}%")
print(f"\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

# Directional F1
_, _, per_f1, _ = precision_recall_fscore_support(all_labels, all_preds, labels=[0,1,2], zero_division=0)
dir_f1 = (per_f1[0] + per_f1[1]) / 2
print(f"Directional F1 (BUY+SELL avg): {dir_f1:.4f}  (BUY={per_f1[0]:.4f}, SELL={per_f1[1]:.4f})")

# BUY/SELL only accuracy
trade_mask = np.isin(all_labels, [0, 1])
if trade_mask.sum() > 0:
    trade_acc = (all_preds[trade_mask] == all_labels[trade_mask]).mean()
    print(f"BUY/SELL accuracy: {trade_acc:.2%} ({trade_mask.sum():,} samples)")

# Confusion matrix & confidence distribution
cm = confusion_matrix(all_labels, all_preds)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix — WaveFollower Test Set")

for i, name in enumerate(CLASS_NAMES):
    mask = all_preds == i
    if mask.sum() > 0:
        axes[1].hist(all_confs[mask], bins=30, alpha=0.6, label=name)
axes[1].set_title("Confidence Distribution by Signal")
axes[1].set_xlabel("Confidence")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Calibration curve
correct_mask = (all_preds == all_labels)
bins = np.linspace(0, 1, 11)
bin_acc, bin_conf = [], []
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (all_confs >= lo) & (all_confs < hi)
    if m.sum() > 0:
        bin_acc.append(correct_mask[m].mean())
        bin_conf.append(all_confs[m].mean())

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect")
ax.plot(bin_conf, bin_acc, "o-", color="#2196F3", label="WaveFollower")
ax.set_title("Confidence Calibration")
ax.set_xlabel("Predicted confidence")
ax.set_ylabel("Actual accuracy")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Trend distribution
from collections import Counter
trend_counts = Counter(all_trends)
print(f"\nTrend predictions: UP={trend_counts.get(0,0):,}  DOWN={trend_counts.get(1,0):,}  NEUTRAL={trend_counts.get(2,0):,}")

## 6. Realistic Backtest — $100 Start, 10% Risk, OANDA Conditions

Runs a separate backtest per pair with pair-specific OANDA spreads and pip values, then aggregates.

In [ ]:
from wavetrader.backtest import run_backtest
from wavetrader.config import BacktestConfig

INITIAL_BALANCE = 100.0
RISK_PER_TRADE = 0.10  # 10% — aggressive for small account growth

# Run backtest per pair with OANDA-realistic settings
all_results = {}

for pair_tag in test_data:
    pair_name = PAIR_NAMES[pair_tag]
    spread = OANDA_SPREADS.get(pair_tag, 2.5)
    pip_val = PIP_VALUES.get(pair_tag, 6.5)
    
    bt_config = BacktestConfig(
        initial_balance=INITIAL_BALANCE,
        risk_per_trade=RISK_PER_TRADE,
        min_confidence=0.55,           # Only trade when model is confident
        spread_pips=spread,            # OANDA typical spread for this pair
        pip_value=pip_val,             # USD per pip per standard lot
        commission_per_lot=0.0,        # OANDA Standard = no commission
        leverage=30.0,                 # ESMA retail leverage cap
        atr_halt_multiplier=3.0,       # Halt on volatility spikes
        drawdown_reduce_threshold=0.15, # Halve risk at 15% drawdown
    )
    
    # WaveFollower config needs pair set for backtest
    bt_model_config = WaveFollowerConfig(
        timeframes=config.timeframes,
        lookbacks=config.lookbacks,
        tf_wave_dim=config.tf_wave_dim,
        fused_dim=config.fused_dim,
        predictor_layers=config.predictor_layers,
        predictor_heads=config.predictor_heads,
        dropout=config.dropout,
        pair=pair_name,
    )
    
    print(f"\n{'='*60}")
    print(f"  Backtesting {pair_name} (spread={spread} pips, pip_val=${pip_val})")
    print(f"{'='*60}")
    
    results = run_backtest(model, test_data[pair_tag], bt_model_config, bt_config, device)
    all_results[pair_tag] = (results, bt_config)

# ── Aggregate summary ─────────────────────────────────────────────────────────
print(f"\n\n{'='*70}")
print(f"  AGGREGATE BACKTEST — WaveFollower ($100 start, 10% risk, OANDA)")
print(f"{'='*70}")
print(f"{'Pair':>10} | {'Trades':>7} | {'Win Rate':>8} | {'PF':>6} | {'Final $':>10} | {'ROI':>8} | {'MaxDD':>7} | {'Sharpe':>7}")
print(f"{'-'*70}")

total_trades = 0
for pair_tag in all_results:
    r, bc = all_results[pair_tag]
    roi = (r.final_balance / bc.initial_balance - 1) * 100
    print(f"{PAIR_NAMES[pair_tag]:>10} | {r.total_trades:>7} | {r.win_rate:>7.1%} | {r.profit_factor:>6.2f} | "
          f"${r.final_balance:>9,.2f} | {roi:>+7.1f}% | {r.max_drawdown:>6.1%} | {r.sharpe_ratio:>7.2f}")
    total_trades += r.total_trades

print(f"{'-'*70}")
print(f"Total trades across all pairs: {total_trades}")

## 7. Primary Pair Deep Dive — GBP/JPY Trade Breakdown

Full trade log, period analysis, session breakdown for the primary pair.

In [ ]:
# ── Select primary pair for deep analysis ──────────────────────────────────────
PRIMARY_PAIR = "GBPJPY"  # Change to analyze a different pair

if PRIMARY_PAIR not in all_results:
    PRIMARY_PAIR = list(all_results.keys())[0]

results, bt_config = all_results[PRIMARY_PAIR]
trades = results.trades
pair_name = PAIR_NAMES[PRIMARY_PAIR]

print(f"Deep dive: {pair_name} — {len(trades)} closed trades")

# Build trade DataFrame
rows = []
for t in trades:
    direction = "BUY" if t.direction.value == 0 else "SELL"
    rows.append({
        "entry_time": t.entry_time, "exit_time": t.exit_time,
        "direction": direction, "entry_price": t.entry_price,
        "exit_price": t.exit_price, "stop_loss": t.stop_loss,
        "take_profit": t.take_profit, "size_lots": t.size,
        "pnl": t.pnl, "exit_reason": t.exit_reason,
        "is_winner": t.pnl > 0,
    })

trade_df = pd.DataFrame(rows)
trade_df["entry_time"] = pd.to_datetime(trade_df["entry_time"])
trade_df["exit_time"] = pd.to_datetime(trade_df["exit_time"])
trade_df["cumulative_pnl"] = trade_df["pnl"].cumsum()
trade_df["trade_num"] = range(1, len(trade_df) + 1)
trade_df["duration"] = trade_df["exit_time"] - trade_df["entry_time"]
trade_df["balance_after"] = bt_config.initial_balance + trade_df["cumulative_pnl"]
trade_df["running_win_rate"] = trade_df["is_winner"].expanding().mean()

print(f"Date range: {trade_df['entry_time'].min()} → {trade_df['exit_time'].max()}")
print(f"\n── First 5 Trades ──")
display(trade_df.head())
print(f"\n── Last 5 Trades ──")
display(trade_df.tail())

In [ ]:
# ── Period Breakdown: Daily / Weekly / Monthly / Yearly ────────────────────────

def period_stats(group):
    wins = group["is_winner"].sum()
    losses = len(group) - wins
    gp = group.loc[group["pnl"] > 0, "pnl"].sum()
    gl = abs(group.loc[group["pnl"] <= 0, "pnl"].sum())
    return pd.Series({
        "trades": len(group), "wins": int(wins), "losses": int(losses),
        "win_rate": wins / max(len(group), 1),
        "gross_profit": gp, "gross_loss": gl,
        "net_pnl": group["pnl"].sum(), "avg_pnl": group["pnl"].mean(),
        "max_win": group["pnl"].max(), "max_loss": group["pnl"].min(),
        "profit_factor": gp / max(gl, 1e-9),
        "avg_duration": group["duration"].mean(),
    })

trade_df["exit_date"] = trade_df["exit_time"].dt.date
daily = trade_df.groupby("exit_date").apply(period_stats).reset_index()
daily["exit_date"] = pd.to_datetime(daily["exit_date"])
daily["cumulative_pnl"] = daily["net_pnl"].cumsum()

trade_df["exit_week"] = trade_df["exit_time"].dt.to_period("W")
weekly = trade_df.groupby("exit_week").apply(period_stats).reset_index()
weekly["cumulative_pnl"] = weekly["net_pnl"].cumsum()

trade_df["exit_month"] = trade_df["exit_time"].dt.to_period("M")
monthly = trade_df.groupby("exit_month").apply(period_stats).reset_index()
monthly["cumulative_pnl"] = monthly["net_pnl"].cumsum()

trade_df["exit_year"] = trade_df["exit_time"].dt.year
yearly = trade_df.groupby("exit_year").apply(period_stats).reset_index()
yearly["cumulative_pnl"] = yearly["net_pnl"].cumsum()

print("=" * 80)
print(f"  YEARLY BREAKDOWN — {pair_name}")
print("=" * 80)
for _, r in yearly.iterrows():
    print(f"  {int(r['exit_year'])}  |  Trades: {int(r['trades']):>5}  |  "
          f"W/L: {int(r['wins'])}/{int(r['losses'])}  |  WR: {r['win_rate']:.1%}  |  "
          f"PnL: ${r['net_pnl']:>+12,.2f}  |  PF: {r['profit_factor']:.2f}")

print(f"\n{'='*80}")
print(f"  MONTHLY BREAKDOWN")
print("=" * 80)
for _, r in monthly.iterrows():
    print(f"  {str(r['exit_month']):>7}  |  Trades: {int(r['trades']):>4}  |  "
          f"W/L: {int(r['wins']):>3}/{int(r['losses']):<3}  |  WR: {r['win_rate']:.0%}  |  "
          f"PnL: ${r['net_pnl']:>+10,.2f}  |  PF: {r['profit_factor']:>5.2f}")

print(f"\n{'='*80}")
print(f"  WEEKLY SUMMARY ({len(weekly)} weeks)")
print("=" * 80)
print(f"  Profitable weeks : {(weekly['net_pnl'] > 0).sum()} / {len(weekly)}  ({(weekly['net_pnl'] > 0).mean():.1%})")
print(f"  Best week PnL    : ${weekly['net_pnl'].max():>+12,.2f}")
print(f"  Worst week PnL   : ${weekly['net_pnl'].min():>+12,.2f}")
print(f"  Avg week PnL     : ${weekly['net_pnl'].mean():>+12,.2f}")

print(f"\n{'='*80}")
print(f"  DAILY SUMMARY ({len(daily)} trading days)")
print("=" * 80)
print(f"  Profitable days  : {(daily['net_pnl'] > 0).sum()} / {len(daily)}  ({(daily['net_pnl'] > 0).mean():.1%})")
print(f"  Best day PnL     : ${daily['net_pnl'].max():>+12,.2f}")
print(f"  Worst day PnL    : ${daily['net_pnl'].min():>+12,.2f}")
print(f"  Avg trades/day   : {daily['trades'].mean():.1f}")

In [ ]:
# ── Full Week-by-Week Breakdown — GBP/JPY ─────────────────────────────────────
print("=" * 95)
print(f"  WEEK-BY-WEEK — {pair_name}  ({len(weekly)} weeks)")
print("=" * 95)
print(f"  {'Week':>12}  |  {'Trades':>6}  |  {'W/L':>7}  |  {'WR':>5}  |  {'Net PnL':>12}  |  {'Cum PnL':>12}  |  {'PF':>5}")
print(f"  {'-'*12}  +  {'-'*6}  +  {'-'*7}  +  {'-'*5}  +  {'-'*12}  +  {'-'*12}  +  {'-'*5}")

for _, r in weekly.iterrows():
    week_str = str(r["exit_week"])
    cum = r["cumulative_pnl"]
    marker = "  " if r["net_pnl"] >= 0 else " *"
    print(f"{marker}{week_str:>12}  |  {int(r['trades']):>6}  |  {int(r['wins']):>3}/{int(r['losses']):<3}  |  "
          f"{r['win_rate']:>4.0%}  |  ${r['net_pnl']:>+11,.2f}  |  ${cum:>+11,.2f}  |  {r['profit_factor']:>5.2f}")

# Summary at bottom
profit_wks = (weekly["net_pnl"] > 0).sum()
loss_wks = (weekly["net_pnl"] <= 0).sum()
print("=" * 95)
print(f"  Profitable: {profit_wks}/{len(weekly)} ({profit_wks/len(weekly):.0%})  |  "
      f"Loss: {loss_wks}/{len(weekly)} ({loss_wks/len(weekly):.0%})  |  "
      f"Avg: ${weekly['net_pnl'].mean():>+,.2f}/wk  |  "
      f"Median: ${weekly['net_pnl'].median():>+,.2f}/wk")
print(f"  Best:  ${weekly['net_pnl'].max():>+12,.2f}  |  Worst: ${weekly['net_pnl'].min():>+12,.2f}  |  "
      f"Std: ${weekly['net_pnl'].std():>,.2f}")
print("=" * 95)

In [ ]:
# ── Timeline Visualizations ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 1, figsize=(16, 20))
fig.suptitle(f"WaveFollower Trade Analysis — {pair_name} ($100 start, 10% risk)",
             fontsize=16, fontweight="bold", y=0.98)

# 1. Per-trade PnL + cumulative
ax = axes[0]
colors = ["#4CAF50" if p > 0 else "#F44336" for p in trade_df["pnl"]]
ax.bar(trade_df["trade_num"], trade_df["pnl"], color=colors, width=1.0, alpha=0.5, label="Per-trade PnL")
ax2 = ax.twinx()
ax2.plot(trade_df["trade_num"], trade_df["balance_after"], color="#2196F3", lw=1.5, label="Balance")
ax2.set_ylabel("Balance ($)")
ax.set_xlabel("Trade #")
ax.set_ylabel("PnL ($)")
ax.set_title("Per-Trade PnL & Balance Growth")
ax.grid(alpha=0.2)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

# 2. Monthly PnL bars
ax = axes[1]
month_labels = [str(m) for m in monthly["exit_month"]]
monthly_colors = ["#4CAF50" if p >= 0 else "#F44336" for p in monthly["net_pnl"]]
ax.bar(range(len(monthly)), monthly["net_pnl"], color=monthly_colors, edgecolor="white", lw=0.5)
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(month_labels, rotation=45, ha="right", fontsize=8)
ax.axhline(0, color="gray", lw=0.8)
ax.set_ylabel("Net PnL ($)")
ax.set_title("Monthly Net PnL")
ax.grid(alpha=0.2, axis="y")

# 3. Rolling win rate
ax = axes[2]
window = min(50, len(trade_df) // 5) or 10
rolling_wr = trade_df["is_winner"].rolling(window).mean() * 100
ax.plot(trade_df["trade_num"], rolling_wr, color="#FF9800", lw=1.2, label=f"{window}-trade WR")
ax.axhline(50, linestyle="--", color="gray", alpha=0.6)
ax.fill_between(trade_df["trade_num"], 50, rolling_wr,
                where=rolling_wr >= 50, color="#4CAF50", alpha=0.15)
ax.fill_between(trade_df["trade_num"], 50, rolling_wr,
                where=rolling_wr < 50, color="#F44336", alpha=0.15)
ax.set_xlabel("Trade #")
ax.set_ylabel("Win Rate (%)")
ax.set_title(f"Rolling Win Rate ({window}-trade window)")
ax.legend()
ax.grid(alpha=0.2)
ax.set_ylim(0, 100)

# 4. PnL by exit reason
ax = axes[3]
reason_stats = trade_df.groupby("exit_reason").agg(
    count=("pnl", "size"), total_pnl=("pnl", "sum"),
    win_rate=("is_winner", "mean"),
).sort_values("count", ascending=True)
bar_colors = ["#4CAF50" if p >= 0 else "#F44336" for p in reason_stats["total_pnl"]]
y = range(len(reason_stats))
ax.barh(y, reason_stats["total_pnl"], color=bar_colors, edgecolor="white")
ax.set_yticks(y)
ax.set_yticklabels([f"{r}\n({int(c)} trades, {wr:.0%})" for r, c, wr
                    in zip(reason_stats.index, reason_stats["count"], reason_stats["win_rate"])])
ax.set_xlabel("Total PnL ($)")
ax.set_title("PnL by Exit Reason")
ax.axvline(0, color="gray", lw=0.8)
ax.grid(alpha=0.2, axis="x")

plt.tight_layout()
plt.show()

# Duration distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
dur_hours = trade_df["duration"].dt.total_seconds() / 3600
axes[0].hist(dur_hours, bins=50, color="#9C27B0", alpha=0.7, edgecolor="white")
axes[0].set_xlabel("Duration (hours)")
axes[0].set_title("Trade Duration Distribution")
axes[0].grid(alpha=0.2)

win_dur = dur_hours[trade_df["is_winner"]]
loss_dur = dur_hours[~trade_df["is_winner"]]
axes[1].hist(win_dur, bins=40, alpha=0.6, color="#4CAF50", label=f"Winners ({win_dur.mean():.1f}h)")
axes[1].hist(loss_dur, bins=40, alpha=0.6, color="#F44336", label=f"Losers ({loss_dur.mean():.1f}h)")
axes[1].set_xlabel("Duration (hours)")
axes[1].set_title("Duration: Winners vs Losers")
axes[1].legend()
axes[1].grid(alpha=0.2)
plt.tight_layout()
plt.show()

# Direction breakdown
print(f"\n{'='*70}")
print(f"  DIRECTION BREAKDOWN — {pair_name}")
print("=" * 70)
for direction in ["BUY", "SELL"]:
    d = trade_df[trade_df["direction"] == direction]
    if len(d) == 0: continue
    wins = d["is_winner"].sum()
    gp = d.loc[d["pnl"] > 0, "pnl"].sum()
    gl = abs(d.loc[d["pnl"] <= 0, "pnl"].sum())
    print(f"  {direction:>4}  |  Trades: {len(d):>5}  |  W/L: {int(wins)}/{len(d)-int(wins)}  |  "
          f"WR: {wins/len(d):.1%}  |  Net: ${d['pnl'].sum():>+12,.2f}  |  PF: {gp/max(gl,1e-9):.2f}")

# Streak analysis
streaks = []
cs = 0
for w in trade_df["is_winner"]:
    cs = max(0, cs) + 1 if w else min(0, cs) - 1
    streaks.append(cs)
trade_df["streak"] = streaks
print(f"\n  Max winning streak : {max(streaks)} trades")
print(f"  Max losing streak  : {abs(min(streaks))} trades")

In [ ]:
# ── Session & Time-of-Day Analysis ─────────────────────────────────────────────
trade_df["entry_hour"] = trade_df["entry_time"].dt.hour
trade_df["entry_dow"] = trade_df["entry_time"].dt.day_name()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f"WaveFollower Session Analysis — {pair_name}", fontsize=14, fontweight="bold")

# 1. Trades by hour
ax = axes[0, 0]
hourly = trade_df.groupby("entry_hour").agg(
    trades=("pnl", "size"), net_pnl=("pnl", "sum"), win_rate=("is_winner", "mean"),
).reindex(range(24), fill_value=0)
bar_colors = ["#4CAF50" if p >= 0 else "#F44336" for p in hourly["net_pnl"]]
ax.bar(hourly.index, hourly["trades"], color=bar_colors, edgecolor="white", alpha=0.8)
ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("Trades")
ax.set_title("Trade Count by Hour")
ax.axvspan(0, 8, alpha=0.05, color="blue", label="Asia")
ax.axvspan(7, 16, alpha=0.05, color="green", label="London")
ax.axvspan(13, 22, alpha=0.05, color="orange", label="NY")
ax.legend(fontsize=8)

# 2. PnL by hour
ax = axes[0, 1]
bar_colors = ["#4CAF50" if p >= 0 else "#F44336" for p in hourly["net_pnl"]]
ax.bar(hourly.index, hourly["net_pnl"], color=bar_colors, edgecolor="white")
ax2 = ax.twinx()
ax2.plot(hourly.index, hourly["win_rate"] * 100, "o-", color="#FF9800", ms=4, lw=1.5)
ax2.set_ylabel("Win Rate (%)")
ax2.set_ylim(0, 100)
ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("Net PnL ($)")
ax.set_title("PnL & Win Rate by Hour")
ax.axhline(0, color="gray", lw=0.8)
ax.grid(alpha=0.2)

# 3. Day of week PnL
ax = axes[1, 0]
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
dow = trade_df.groupby("entry_dow").agg(
    trades=("pnl", "size"), net_pnl=("pnl", "sum"), win_rate=("is_winner", "mean"),
).reindex(dow_order).dropna()
bar_colors = ["#4CAF50" if p >= 0 else "#F44336" for p in dow["net_pnl"]]
x = range(len(dow))
ax.bar(x, dow["net_pnl"], color=bar_colors, edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels([d[:3] for d in dow.index])
ax.set_ylabel("Net PnL ($)")
ax.set_title("Net PnL by Day of Week")
ax.axhline(0, color="gray", lw=0.8)
ax.grid(alpha=0.2, axis="y")

# 4. Win rate by day of week
ax = axes[1, 1]
ax.bar(x, dow["win_rate"] * 100, color="#2196F3", edgecolor="white", alpha=0.8)
ax.axhline(50, linestyle="--", color="gray", alpha=0.6)
for i, (wr, n) in enumerate(zip(dow["win_rate"], dow["trades"])):
    ax.text(i, wr * 100 + 1.5, f"{int(n)}t", ha="center", fontsize=9, color="gray")
ax.set_xticks(x)
ax.set_xticklabels([d[:3] for d in dow.index])
ax.set_ylabel("Win Rate (%)")
ax.set_title("Win Rate by Day")
ax.set_ylim(0, 100)
ax.grid(alpha=0.2, axis="y")

plt.tight_layout()
plt.show()

# Session breakdown
def _session(h):
    if 0 <= h < 8: return "Asia (00-08)"
    elif 8 <= h < 13: return "London (08-13)"
    elif 13 <= h < 17: return "Overlap (13-17)"
    else: return "New York (17-22)"

trade_df["session"] = trade_df["entry_hour"].apply(_session)
session_stats = trade_df.groupby("session").agg(
    trades=("pnl", "size"), net_pnl=("pnl", "sum"),
    win_rate=("is_winner", "mean"), avg_pnl=("pnl", "mean"),
)

print(f"\n{'='*70}")
print(f"  SESSION BREAKDOWN — {pair_name}")
print("=" * 70)
for sess, r in session_stats.iterrows():
    print(f"  {sess:<20}  |  Trades: {int(r['trades']):>5}  |  "
          f"WR: {r['win_rate']:.1%}  |  Net: ${r['net_pnl']:>+10,.2f}  |  "
          f"Avg: ${r['avg_pnl']:>+8,.2f}")
print("=" * 70)

## 8. Realistic Friction Simulation — What Would OANDA Actually Return?

Apply real-world OANDA friction: slippage, off-hours spread widening, news spikes, micro-lot size limits.

In [ ]:
# ── Realistic Friction for $100 OANDA Account ─────────────────────────────────
np.random.seed(42)

# OANDA-specific friction parameters
SLIPPAGE_RANGE = (0.3, 2.0)     # OANDA execution is good, 0.3-2 pips
SPREAD_OFFHOURS = 1.5           # Extra pips during Asia for GBP/JPY
SPREAD_NEWS_PROB = 0.05         # 5% chance of news spike
SPREAD_NEWS_EXTRA = 4.0         # Spread blowout during news
LOT_CAP = 0.5                   # Realistic cap for $100 account at 30:1
LATENCY_MISS_RATE = 0.02        # 2% missed on OANDA (better than most)
PIP_VALUE = PIP_VALUES.get(PRIMARY_PAIR, 6.5)

print("=" * 70)
print("  OANDA FRICTION SIMULATION")
print("=" * 70)
print(f"  Account         : $100 OANDA (101-001-30902818-002)")
print(f"  Slippage        : {SLIPPAGE_RANGE[0]}-{SLIPPAGE_RANGE[1]} pips")
print(f"  Off-hours spread: +{SPREAD_OFFHOURS} pips")
print(f"  News spike      : {SPREAD_NEWS_PROB:.0%} at +{SPREAD_NEWS_EXTRA} pips")
print(f"  Lot cap         : {LOT_CAP} lots max")
print(f"  Latency misses  : {LATENCY_MISS_RATE:.0%}")
print("=" * 70)

realistic_rows = []
balance_theo = bt_config.initial_balance
balance_real = bt_config.initial_balance

for _, t in trade_df.iterrows():
    balance_theo += t["pnl"]
    
    if np.random.random() < LATENCY_MISS_RATE:
        realistic_rows.append({
            "trade_num": t["trade_num"], "original_pnl": t["pnl"],
            "adjusted_pnl": 0, "skipped": True,
            "slippage_cost": 0, "spread_cost": 0, "lot_penalty": 0,
            "balance_theo": balance_theo, "balance_real": balance_real,
        })
        continue
    
    slippage = np.random.uniform(*SLIPPAGE_RANGE)
    extra_spread = 0.0
    hour = t["entry_hour"] if "entry_hour" in t.index else 12
    if hour < 7 or hour >= 21:
        extra_spread += SPREAD_OFFHOURS
    if np.random.random() < SPREAD_NEWS_PROB:
        extra_spread += SPREAD_NEWS_EXTRA
    
    lot_ratio = min(1.0, LOT_CAP / max(t["size_lots"], 0.001))
    friction_cost = (slippage + extra_spread) * PIP_VALUE * min(t["size_lots"], LOT_CAP)
    adjusted_pnl = (t["pnl"] * lot_ratio) - friction_cost
    balance_real += adjusted_pnl
    
    realistic_rows.append({
        "trade_num": t["trade_num"], "original_pnl": t["pnl"],
        "adjusted_pnl": adjusted_pnl, "skipped": False,
        "slippage_cost": slippage * PIP_VALUE * min(t["size_lots"], LOT_CAP),
        "spread_cost": extra_spread * PIP_VALUE * min(t["size_lots"], LOT_CAP),
        "lot_penalty": t["pnl"] * (1 - lot_ratio),
        "balance_theo": balance_theo, "balance_real": balance_real,
    })

rf = pd.DataFrame(realistic_rows)
active = rf[~rf["skipped"]]

theo_roi = (balance_theo / bt_config.initial_balance - 1) * 100
real_roi = (balance_real / bt_config.initial_balance - 1) * 100
real_wins = (active["adjusted_pnl"] > 0).sum()
real_total = len(active)
real_wr = real_wins / max(real_total, 1)
real_gp = active.loc[active["adjusted_pnl"] > 0, "adjusted_pnl"].sum()
real_gl = abs(active.loc[active["adjusted_pnl"] <= 0, "adjusted_pnl"].sum())
real_pf = real_gp / max(real_gl, 1e-9)

# Estimate annualized (test set is ~21 months of data)
test_months = max(1, (trade_df["exit_time"].max() - trade_df["entry_time"].min()).days / 30)
real_annual = ((balance_real / bt_config.initial_balance) ** (12 / test_months) - 1) * 100

print(f"\n{'='*70}")
print(f"  {'METRIC':<25} {'THEORETICAL':>15} {'REALISTIC':>15}")
print("=" * 70)
print(f"  {'Trades Executed':<25} {len(trade_df):>15,} {real_total:>15,}")
print(f"  {'Trades Skipped':<25} {'0':>15} {rf['skipped'].sum():>15,}")
print(f"  {'Final Balance':<25} ${balance_theo:>14,.2f} ${balance_real:>14,.2f}")
print(f"  {'Total ROI':<25} {theo_roi:>+14.1f}% {real_roi:>+14.1f}%")
print(f"  {'Annualized ROI':<25} {'—':>15} {real_annual:>+14.1f}%")
print(f"  {'Win Rate':<25} {results.win_rate:>14.1%} {real_wr:>14.1%}")
print(f"  {'Profit Factor':<25} {results.profit_factor:>15.2f} {real_pf:>15.2f}")
print(f"  {'Total Friction Cost':<25} {'$0':>15} ${(active['slippage_cost'].sum() + active['spread_cost'].sum()):>14,.2f}")
print("=" * 70)

# Compound projections
print(f"\n  Projections from $100 at {real_annual:+.0f}% annualized:")
for yr in [1, 2, 3, 5]:
    projected = bt_config.initial_balance * (1 + real_annual / 100) ** yr
    print(f"    Year {yr}: ${projected:>12,.2f}")

# Equity curves
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

axes[0].plot(rf["trade_num"], rf["balance_theo"], lw=1.2, color="#2196F3", alpha=0.6, label="Theoretical")
axes[0].plot(rf["trade_num"], rf["balance_real"], lw=1.5, color="#4CAF50", label="Realistic (OANDA)")
axes[0].axhline(bt_config.initial_balance, ls="--", color="gray", alpha=0.5)
axes[0].set_ylabel("Balance ($)")
axes[0].set_title("Theoretical vs Realistic Equity — WaveFollower on OANDA")
axes[0].legend()
axes[0].grid(alpha=0.2)

axes[1].plot(rf["trade_num"], rf["balance_theo"], lw=1.2, color="#2196F3", alpha=0.6, label="Theoretical")
axes[1].plot(rf["trade_num"], rf["balance_real"], lw=1.5, color="#4CAF50", label="Realistic")
axes[1].set_yscale("log")
axes[1].set_xlabel("Trade #")
axes[1].set_ylabel("Balance ($) — log")
axes[1].set_title("Log Scale (straight = consistent compounding)")
axes[1].legend()
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 9. Save All Results to Google Drive

In [ ]:
# ── Save everything ───────────────────────────────────────────────────────────
SAVE_DIR = DRIVE_ROOT / "backtest_results" / CKPT_DIR.name
os.makedirs(SAVE_DIR, exist_ok=True)

# Trade log
trade_export = trade_df.drop(columns=["exit_date", "exit_week", "exit_month", "exit_year", "streak"], errors="ignore")
trade_export.to_csv(SAVE_DIR / "trade_log.csv", index=False)
print(f"Saved trade_log.csv           ({len(trade_export)} trades)")

# Period breakdowns
daily.to_csv(SAVE_DIR / "daily_breakdown.csv", index=False)
print(f"Saved daily_breakdown.csv     ({len(daily)} days)")

weekly_export = weekly.copy()
weekly_export["exit_week"] = weekly_export["exit_week"].astype(str)
weekly_export.to_csv(SAVE_DIR / "weekly_breakdown.csv", index=False)
print(f"Saved weekly_breakdown.csv    ({len(weekly_export)} weeks)")

monthly_export = monthly.copy()
monthly_export["exit_month"] = monthly_export["exit_month"].astype(str)
monthly_export.to_csv(SAVE_DIR / "monthly_breakdown.csv", index=False)
print(f"Saved monthly_breakdown.csv   ({len(monthly_export)} months)")

yearly.to_csv(SAVE_DIR / "yearly_breakdown.csv", index=False)
print(f"Saved yearly_breakdown.csv    ({len(yearly)} years)")

# Equity curve
eq_df = pd.DataFrame({"equity": results.equity_curve})
eq_df.to_csv(SAVE_DIR / "equity_curve.csv", index=False)
print(f"Saved equity_curve.csv        ({len(eq_df)} points)")

# Session stats
session_stats.to_csv(SAVE_DIR / "session_breakdown.csv")
print(f"Saved session_breakdown.csv   ({len(session_stats)} sessions)")

# Friction simulation
rf.to_csv(SAVE_DIR / "friction_simulation.csv", index=False)
print(f"Saved friction_simulation.csv ({len(rf)} rows)")

# Per-pair aggregate summary
agg_rows = []
for pt in all_results:
    r, bc = all_results[pt]
    agg_rows.append({
        "pair": PAIR_NAMES[pt], "trades": r.total_trades,
        "win_rate": r.win_rate, "profit_factor": r.profit_factor,
        "final_balance": r.final_balance,
        "roi_pct": (r.final_balance / bc.initial_balance - 1) * 100,
        "max_drawdown": r.max_drawdown, "sharpe": r.sharpe_ratio,
    })
pd.DataFrame(agg_rows).to_csv(SAVE_DIR / "per_pair_summary.csv", index=False)
print(f"Saved per_pair_summary.csv    ({len(agg_rows)} pairs)")

print(f"\nAll results saved to: {SAVE_DIR}")
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(SAVE_DIR / f)
    print(f"  {f:<35s}  {size/1024:>8.1f} KB")

In [ ]:
# ── Equity Curve & Sample Trades ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
equity = results.equity_curve

ax.plot(equity, lw=1.2, color="#4CAF50")
ax.axhline(bt_config.initial_balance, ls="--", color="gray", alpha=0.7, label="$100 start")
ax.fill_between(range(len(equity)), bt_config.initial_balance, equity,
                where=[e > bt_config.initial_balance for e in equity],
                color="#4CAF50", alpha=0.15)
ax.fill_between(range(len(equity)), bt_config.initial_balance, equity,
                where=[e < bt_config.initial_balance for e in equity],
                color="#F44336", alpha=0.15)
ax.set_title(f"WaveFollower Equity Curve — {pair_name} ($100 start, 10% risk, OANDA)")
ax.set_xlabel("Bars")
ax.set_ylabel("Balance ($)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nSample Trades (first 15):")
print(f"  {'Dir':>5}  {'Entry':>10}  {'Exit':>10}  {'PnL':>10}  {'Reason':>14}  {'Win':>4}")
print(f"  {'─'*5}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*14}  {'─'*4}")
for _, row in trade_df.head(15).iterrows():
    win = "✓" if row["is_winner"] else "✗"
    print(f"  {row['direction']:>5}  {row['entry_price']:>10.3f}  {row['exit_price']:>10.3f}  "
          f"${row['pnl']:>+9.2f}  {row['exit_reason']:>14}  {win:>4}")